# Regresión con métodos de restricción (Ridge y Lasso) — datos de venta de inmuebles

**Objetivo:** aplicar y comparar OLS, Ridge y Lasso sobre datos de venta de inmuebles en la Ciudad de México y municipios conurbados, para ilustrar qué hacen realmente los métodos de *shrinkage* (regresiones restringidas) frente a mínimos cuadrados ordinarios — el ejemplo central del curso para la técnica de LASSO.


## 1. Dependencias y carga de datos

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

In [ ]:
# Los datos se cargan desde un ZIP local para mantener el repo ligero.
# Venta_sel.csv ya es una versión filtrada del scraping original (Ventas.csv, no incluido
# por su tamaño) que conserva solo anuncios con precio, superficie, recámaras y baños completos.
with zipfile.ZipFile('Venta_sel.csv.zip', 'r') as z:
    with z.open('Venta_sel.csv') as f:
        Datos = pd.read_csv(f, encoding='utf-8-sig')

print(Datos.shape)
Datos = Datos.rename(columns={'construction (m2)': 'Construccion_m2', 'terrain (m2)': 'Terreno_m2'})
Datos[['Monto1', 'rooms', 'bathrooms', 'Construccion_m2', 'Terreno_m2', 'propiedad', 'ciudad']].head()

## 2. Limpieza

El scraping trae varios problemas típicos de web scraping: nombres de ciudad inconsistentes,
anuncios duplicados, columnas sin cobertura suficiente para usarse como feature, y valores de
precio que son casi con certeza errores de captura. Cada paso de esta sección se cuantifica
(cuántas filas o columnas cambia) para que el efecto de la limpieza quede explícito, no implícito.

### 2.1 Ciudades

`ciudad` trae "Ciudad de México" y "Mexico City" como si fueran lugares distintos (problema
típico de geocodificación), y **1,581 anuncios con `ciudad` vacío** (texto vacío, no `NA`) que se
colarían entre las "ciudades" más frecuentes si no los filtramos primero.

In [ ]:
Datos['ciudad'] = Datos['ciudad'].replace('Mexico City', 'Ciudad de México')
Datos = Datos[Datos['ciudad'] != '']

top_ciudades = Datos['ciudad'].value_counts().head(4).index.tolist()
Datos = Datos[Datos['ciudad'].isin(top_ciudades)]

print('Ciudades:', top_ciudades)
print('Filas tras limpieza de ciudad:', Datos.shape[0])

### 2.2 Anuncios duplicados

Algunos inmuebles aparecen publicados más de una vez: mismo texto, mismas coordenadas, mismo
precio, pero con un `link` distinto (típico de re-publicaciones, o de un mismo asesor subiendo el
anuncio más de una vez). Si no los quitamos, esas propiedades pesan más de una vez en el ajuste
del modelo — no es un problema de datos faltantes, sino de **observaciones repetidas** que inflan
artificialmente su influencia. Los identificamos comparando todas las columnas excepto `link`
(que por construcción es distinto incluso para el mismo inmueble re-publicado) y nos quedamos con
la primera aparición de cada uno.

In [ ]:
cols_except_link = [c for c in Datos.columns if c != 'link']
n_dup = Datos.duplicated(subset=cols_except_link).sum()
print(f'Anuncios duplicados (mismo inmueble, distinto link): {n_dup}')

Datos = Datos.drop_duplicates(subset=cols_except_link)
print('Filas tras quitar duplicados:', Datos.shape[0])

### 2.3 Columnas que no se usan en el modelo

El scraping trae 25 columnas, pero varias no sirven como feature en este ejemplo:

- `calle`, `numero` y `CP` faltan en 55%-83% de los anuncios (ver el conteo abajo) y `colonia`
  falta en cerca de 9% adicional — sin cobertura completa ni una forma consistente de
  codificarlas (habría que geocodificar texto libre), no se pueden usar sin trabajo adicional que
  está fuera del alcance de este ejemplo.
- `Iniciales` identifica al asesor o agencia que publicó el anuncio, no una característica del
  inmueble.
- `operation` tiene un solo valor (`'venta'`, ya viene filtrado así desde el scraping) — sin
  varianza, no aporta nada a un modelo.
- `name`, `description`, `link`, `price`, `location` y `formatad` son texto libre o
  identificadores; `Moneda` y `Monto` ya están resumidos en `Monto1` (la versión en pesos, con
  conversión de USD incluida, que sí usamos más adelante).

Las quitamos explícitamente — no solo omitirlas del feature set más adelante — para que quede
documentado qué información se descarta y por qué, en vez de que el lector tenga que inferirlo de
qué columnas sí se usan.

In [ ]:
print('% de valores faltantes:')
print(Datos[['calle', 'numero', 'colonia', 'CP']].isna().mean().round(3))

cols_sin_uso = ['name', 'description', 'link', 'price', 'operation', 'entidad', 'Moneda',
                'Monto', 'Iniciales', 'calle', 'numero', 'colonia', 'CP', 'formatad', 'location']
Datos = Datos.drop(columns=[c for c in cols_sin_uso if c in Datos.columns])
print('Columnas restantes:', Datos.columns.tolist())

### 2.4 Outliers en precio por m²

`Price_m2` (ya calculado en el scraping como `Monto1 / Construccion_m2`) tiene un mínimo de poco
más de $10 por m² y un máximo de más de 1.3 millones de pesos por m² — ambos extremos son casi
con certeza errores de captura (una casa completa anunciada en $2,250 o $10,000 pesos, o un
departamento con un precio por m² fuera de cualquier rango real del mercado), no inmuebles
genuinamente baratos o caros. Dejarlos entrar distorsiona el ajuste porque son *puntos de
apalancamiento* (leverage points): unas pocas observaciones extremas con peso desproporcionado en
una regresión por mínimos cuadrados.

La regla estándar de outliers por rango intercuartílico (IQR, $[Q_1 - 1.5 \cdot IQR,\ Q_3 + 1.5
\cdot IQR]$) **no funciona bien aquí**: la cola derecha es tan larga que el límite inferior que
produce es negativo (es decir, no filtraría ningún valor bajo, que es justo el problema que
queremos corregir). En su lugar usamos un corte pragmático por percentiles: quitamos el 1% más
bajo y el 1% más alto de `Price_m2`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
Datos.boxplot(column='Price_m2', ax=ax[0])
ax[0].set_title('Antes de filtrar')

q_low, q_high = Datos['Price_m2'].quantile([0.01, 0.99])
print(f'Percentil 1%: {q_low:,.0f} pesos/m2   Percentil 99%: {q_high:,.0f} pesos/m2')

antes = len(Datos)
Datos = Datos[(Datos['Price_m2'] >= q_low) & (Datos['Price_m2'] <= q_high)]
print(f'Outliers removidos: {antes - len(Datos)} de {antes}')

Datos.boxplot(column='Price_m2', ax=ax[1])
ax[1].set_title('Después de filtrar')
plt.tight_layout()
plt.show()

### 2.5 log(Precio de venta)

El precio está fuertemente sesgado a la derecha; trabajamos con log(precio), práctica común en
modelos hedónicos de precios de vivienda.

In [ ]:
Datos['log_Monto'] = np.log(Datos['Monto1'])
Datos['log_Monto'].hist(bins=50, figsize=(6, 4))
plt.title('Distribución de log(Precio de venta)')
plt.xlabel('log(Monto1)')
plt.show()

## 3. Exploración visual

Antes de construir features y ajustar modelos, vale la pena ver cómo se ven los datos ya
limpios.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for tipo, grupo in Datos.groupby('propiedad'):
    ax.scatter(grupo['Construccion_m2'], grupo['log_Monto'], s=10, alpha=0.35, label=tipo)
ax.set_xlabel('Superficie construida (m2)')
ax.set_ylabel('log(Precio de venta)')
ax.set_title('Relación entre superficie construida y precio de venta')
ax.legend(loc='lower right', title='Tipo de inmueble')
plt.tight_layout()
plt.show()

In [ ]:
Datos.boxplot(column='log_Monto', by='propiedad', figsize=(7, 5))
plt.title('Precio de venta por tipo de inmueble')
plt.suptitle('')
plt.xlabel('Tipo de inmueble')
plt.ylabel('log(Precio de venta)')
plt.tight_layout()
plt.show()

## 4. Construcción de features

Para que Ridge y Lasso realmente tengan algo que hacer necesitamos variables correlacionadas
entre sí (si no, con pocas variables y miles de observaciones, OLS ya es estable y la
regularización casi no cambia nada — lo verificamos más abajo). Agregamos términos polinomiales
de grado 2 de las variables numéricas (que por construcción están correlacionados con la
variable original, ej. `Construccion_m2` y `Construccion_m2^2`) más dummies de `propiedad` y
`ciudad`.

In [ ]:
NUMERIC = ['Construccion_m2', 'Terreno_m2', 'rooms', 'bathrooms', 'lat', 'lng']
Datos = Datos.dropna(subset=NUMERIC + ['log_Monto', 'propiedad', 'ciudad'])

poly = PolynomialFeatures(degree=2, include_bias=False)
num_poly = poly.fit_transform(Datos[NUMERIC])
X_poly = pd.DataFrame(num_poly, columns=poly.get_feature_names_out(NUMERIC), index=Datos.index)

dummies = pd.get_dummies(Datos[['propiedad', 'ciudad']], drop_first=True)

X = pd.concat([X_poly, dummies], axis=1)
y = Datos['log_Monto']

print(f'{X.shape[1]} features (de {len(NUMERIC)} numéricas originales)')
print(f'Correlación Construccion_m2 vs Construccion_m2^2: {X[["Construccion_m2", "Construccion_m2^2"]].corr().iloc[0, 1]:.2f}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f'Entrenamiento: {X_train.shape},  Prueba: {X_test.shape}')

## 5. OLS, Ridge y Lasso

Los tres modelos se ajustan sobre las variables estandarizadas (`StandardScaler`), para que la
penalización de Ridge/Lasso trate a todas las variables de forma comparable — de lo contrario,
una variable con escala mucho mayor (por ejemplo `Construccion_m2^2`, en cientos de miles) sería
penalizada de forma muy distinta a una dummy de 0/1.

El valor de penalización $\lambda$ (llamado `alpha` en scikit-learn) de Ridge y Lasso se elige
por validación cruzada (`RidgeCV`, `LassoCV`) en vez de fijarlo a mano.

In [ ]:
linreg = make_pipeline(StandardScaler(), LinearRegression())
linreg.fit(X_train, y_train)

ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 4, 60), cv=5))
ridge.fit(X_train, y_train)

lasso = make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-4, 1, 60), cv=5, max_iter=20000))
lasso.fit(X_train, y_train)

print(f"Ridge — alpha óptimo (CV): {ridge.named_steps['ridgecv'].alpha_:.4f}")
print(f"Lasso — alpha óptimo (CV): {lasso.named_steps['lassocv'].alpha_:.4f}")

## 6. Comparación de desempeño (conjunto de prueba)

In [ ]:
resultados = {}
for nombre, modelo in [('OLS', linreg), ('Ridge', ridge), ('Lasso', lasso)]:
    pred = modelo.predict(X_test)
    resultados[nombre] = {
        'R2_test': r2_score(y_test, pred),
        'MAE_test (escala log)': mean_absolute_error(y_test, pred),
    }

pd.DataFrame(resultados).T

## 7. ¿Qué hace cada método con los coeficientes?

Esta es la comparación que de verdad importa: OLS y Ridge encogen los coeficientes pero
**ningún coeficiente llega exactamente a cero**; Lasso, en cambio, **elimina variables por
completo** (selección de variables), especialmente entre las que están más correlacionadas
entre sí (como los términos cuadráticos y las interacciones con `lat`/`lng`).

In [ ]:
coefs = pd.DataFrame({
    'OLS': linreg.named_steps['linearregression'].coef_,
    'Ridge': ridge.named_steps['ridgecv'].coef_,
    'Lasso': lasso.named_steps['lassocv'].coef_,
}, index=X.columns)

eliminadas = coefs[coefs['Lasso'] == 0].index.tolist()
print(f'Lasso elimina {len(eliminadas)} de {len(coefs)} variables:')
print(eliminadas)

coefs

In [ ]:
top_vars = coefs['OLS'].abs().sort_values(ascending=False).head(15).index
coefs.loc[top_vars].plot.barh(figsize=(8, 8), width=0.8)
plt.title('Coeficientes estandarizados por método (15 variables con mayor efecto en OLS)')
plt.xlabel('Coeficiente (escala estandarizada)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. ¿Y si no hubiera colinealidad?

Repetimos el ejercicio solo con las 6 variables numéricas originales (sin polinomios), donde
prácticamente no hay colinealidad entre ellas. La lección: con pocas variables poco
correlacionadas y miles de observaciones, OLS ya es estable — Ridge y Lasso casi no cambian
nada, porque no hay nada que "corregir".

In [ ]:
X_simple = pd.concat([Datos[NUMERIC], dummies], axis=1)
Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_simple, y, test_size=0.25, random_state=42)

linreg_s = make_pipeline(StandardScaler(), LinearRegression()).fit(Xs_train, ys_train)
lasso_s = make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-4, 1, 60), cv=5, max_iter=20000)).fit(Xs_train, ys_train)

print('R2 OLS  (sin polinomios):', r2_score(ys_test, linreg_s.predict(Xs_test)))
print('R2 Lasso(sin polinomios):', r2_score(ys_test, lasso_s.predict(Xs_test)))
print('Variables eliminadas por Lasso:', (lasso_s.named_steps['lassocv'].coef_ == 0).sum(), '/', X_simple.shape[1])

## Para pensar

- ¿Por qué Ridge nunca lleva un coeficiente exactamente a cero, mientras que Lasso sí puede
  hacerlo? (Pista: piensa en la forma geométrica de las regiones de restricción
  $\sum \beta_k^2 \leq t$ vs. $\sum |\beta_k| \leq t$ de la Sección de Shrinkage Methods.)
- En la sección 8 viste que, sin colinealidad, Ridge/Lasso casi no cambian el resultado de OLS.
  ¿En qué escenarios reales (fuera de este ejemplo) esperarías que la colinealidad entre
  variables sea alta de forma natural, sin tener que construir polinomios a propósito?
- Lasso eliminó variables relacionadas con `lat`/`lng` (ubicación) en su forma cuadrática o de
  interacción, pero no las dummies de `ciudad`. ¿Tiene sentido que conserve la información de
  ubicación en una forma y no en otra?
- Compara el $R^2$ del modelo OLS de la sección 6 (con polinomios, 0.71) contra el $R^2$ del
  OLS sin polinomios de la sección 8 (0.63): ¿por qué mejora tanto solo con agregar términos
  cuadráticos, y por qué eso mismo es lo que le da a Ridge/Lasso algo que "corregir"?
- La sección 2 quitó 332 anuncios duplicados y 665 outliers de `Price_m2` (997 filas en total,
  ~3% del conjunto tras la limpieza de ciudad) antes de ajustar cualquier modelo. El $R^2$ de OLS
  con polinomios subió de 0.67 (versión sin esta limpieza) a 0.71 con ella. ¿Por qué quitar un 3%
  de las filas —las más "raras"— puede mejorar el ajuste sobre el 97% restante, en vez de solo
  reducir el tamaño de muestra? ¿Cómo distinguirías, en un caso real, un outlier que es un error
  de captura de uno que es una observación legítima pero inusual (por ejemplo, una mansión de
  lujo)?